# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR² Mass Spectrometry dataset using `mlcroissant`. This loads the structured Croissant schema as well as available record sets. (See [mlcroissant documentation](https://github.com/mlcommons/croissant) for background.)

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object (not subscripted as a dict, per instructions)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n")

print("Dataset Citation:")
print(metadata.cite_as)

print("\nDate Published:", metadata.date_published)
print("Version:", metadata.version)

print("\nData Collection Description:")
print(metadata.data_collection)

print("\nFields:", metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their `@id`s. The Croissant schema allows us to precisely reference and load data tables or files using these stable IDs.

Let's enumerate the available record sets, fields, and columns in the dataset and view their `@id` structure.

In [ ]:
# List all record sets by their @id
print("Record sets available in dataset:")
record_sets = [rs for rs in metadata.record_sets]
# Print a quick table of all record sets and their fields
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. @id: {rs.id}\n   - Name: {rs.name}  |  Description: {rs.description}")
    print("   Fields:")
    for field in rs.fields:
        print(f"      - Field '@id': {field.id} | Name: {field.name} | Type: {field.data_type}")
    print("")

## 3. Data Extraction
Let's load the data from the major record sets into pandas DataFrames. We'll use each record set's `@id` for precise reference as required.

We'll display the columns of the primary record set and show its first 5 records for inspection.

In [ ]:
# Extract data for all record sets
all_record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs in metadata.record_sets:
    rs_id = rs.id
    try:
        # Some record sets in Croissant may be very large, so display only first few rows
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {rs_id}")
    except Exception as e:
        print(f"Could not load records for '@id': {rs_id} -- {e}")

if dataframes:
    # Display the first, primary record set's columns and head
    primary_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet '@id': {primary_rs_id}")
    print(dataframes[primary_rs_id].columns.tolist())
    dataframes[primary_rs_id].head()
else:
    print("No dataframes were loaded. Please check if data files are public and accessible.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate basic processing: filtering, normalization, and grouping. All field and column references use their Croissant `@id`, as required.

We'll select a numeric field (e.g., 'MW_kDa', mass/weight in kDa units), filter for high molecular weight proteins (>80 kDa), normalize the column, and group by another field (e.g., protein isoelectric point (`pI`), or a categorical group if present).

In [ ]:
import numpy as np

# You may need to consult the field names for accurate @id
primary_rs_id = list(dataframes.keys())[0]
df = dataframes[primary_rs_id]

# Show all columns to help select a numeric field
print(f"Available columns in primary DataFrame ('@id': {primary_rs_id}):\n", df.columns.tolist())

# Choose a likely numeric field by its exact name (case sensitive)
# This field must correspond to a Croissant field @id, often same as the column name in the file
# E.g., 'MW_kDa', 'Abundance', 'Coverage', etc.
possible_num_fields = [col for col in df.columns if df[col].dtype != object]
if not possible_num_fields:
    # Try to infer numeric columns even if dtype is 'object' but contains numbers
    num_like_cols = []
    for col in df.columns:
        try:
            sample = pd.to_numeric(df[col].dropna().iloc[:20])
            if sample.notnull().sum() > 5:
                num_like_cols.append(col)
        except Exception:
            pass
    possible_num_fields = num_like_cols

if not possible_num_fields:
    print('No numeric fields detected. EDA demonstration halted.')
else:
    numeric_field = possible_num_fields[0]  # Use the first found
    print(f"Selected numeric field for filtering and normalization: {numeric_field}")

    # Try to ensure the data is float
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records where '{numeric_field}' > {threshold:.2f} (mean): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field, if any
    group_fields = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        print(f"\nGrouping by '{group_field}' (categorical field):")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print('No suitable categorical fields available for grouping.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field using seaborn/matplotlib. We can also plot the normalized values or a grouped summary if grouping was possible.

> Note: Visualizations depend on at least one numeric field and a successful EDA above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of {numeric_field} (filtered > mean)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True, color='darkorange')
        plt.title(f'Normalized {numeric_field} (Z-score of filtered)')
        plt.xlabel(f'{numeric_field} (Z-score)')
        plt.ylabel('Count')
        plt.show()

    if 'grouped_df' in locals() and group_field in grouped_df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No filtered dataframe available to visualize.')

## 6. Conclusion
- We have successfully loaded metadata, record sets, and tabular data from the FAIR² Mass Spectrometry dataset using Croissant schema and `mlcroissant`.
- Using record set and field `@id`s ensures robust, schema-driven data referencing.
- Basic exploratory analyses (filtering, normalization, grouping) and visualizations reveal the distributional characteristics of quantitative protein features.

**Next steps:**
- Dive deeper by analyzing correlations between fields, protein clusters, or integrating external biological databases.
- For more advanced analysis, consult the Croissant metadata for semantic field descriptions and data provenance.